In [1]:
import urllib.request
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

DATA_DIR = Path.home() / "Downloads" / "transformer_data"
SOURCE_SPLITS = ("Train", "Validation", "Test")
NUM_FRAMES = 37

MODEL_PATH = Path("models/hand_landmarker.task")
SPLIT_FOLDERS = {
    "train": Path("train"),
    "val": Path("validate"),
    "test": Path("test"),
}

_split_lookup = {}
for split_name, csv_path in [
    ("train", "train.csv"),
    ("val", "val.csv"),
    ("test", "test.csv"),
]:
    for video_id in pd.read_csv(csv_path)["video_id"]:
        _split_lookup[video_id] = split_name

_landmarker = None

In [2]:
def find_video(video_id):
    """Return the folder path for a video id under Train, Validation, or Test."""
    for split in SOURCE_SPLITS:
        video_path = DATA_DIR / split / str(video_id)
        if video_path.is_dir():
            return video_path
    raise FileNotFoundError(
        f"Video {video_id} not found in {', '.join(SOURCE_SPLITS)}"
    )


def load_frames(video_path, num_frames=NUM_FRAMES):
    """Load the JPG frames for a video folder."""
    frames = []
    for i in range(1, num_frames + 1):
        frame_path = video_path / f"{i:05d}.jpg"
        frame = cv2.imread(str(frame_path))
        if frame is None:
            raise FileNotFoundError(f"Missing frame: {frame_path}")
        frames.append(frame)
    return frames


def _get_landmarker():
    global _landmarker
    if _landmarker is None:
        MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
        if not MODEL_PATH.exists():
            url = (
                "https://storage.googleapis.com/mediapipe-models/"
                "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
            )
            urllib.request.urlretrieve(url, MODEL_PATH)

        options = vision.HandLandmarkerOptions(
            base_options=python.BaseOptions(model_asset_path=str(MODEL_PATH)),
            running_mode=vision.RunningMode.IMAGE,
            num_hands=1,
            min_hand_detection_confidence=0.3,
        )
        _landmarker = vision.HandLandmarker.create_from_options(options)
    return _landmarker


def extract_landmarks(frames):
    """Run MediaPipe hand landmarks on each frame. Returns shape (37, 63)."""
    landmarker = _get_landmarker()
    landmarks = []

    for frame in frames:
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect(mp_image)

        frame_landmarks = np.zeros((21, 3), dtype=np.float32)
        if result.hand_landmarks:
            for i, lm in enumerate(result.hand_landmarks[0]):
                frame_landmarks[i] = [lm.x, lm.y, lm.z]
        landmarks.append(frame_landmarks)

    landmarks = np.stack(landmarks)
    return landmarks.reshape(NUM_FRAMES, 63)


def save_sequence(video_id, landmarks):
    """Save landmarks to train/, validate/, or test/ based on the split CSVs."""
    split = _split_lookup.get(video_id)
    if split is None:
        raise ValueError(f"video_id {video_id} not found in train.csv, val.csv, or test.csv")

    out_dir = SPLIT_FOLDERS[split]
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / f"{video_id}.npy", landmarks)

In [ ]:
from tqdm import tqdm

for split_folder in SPLIT_FOLDERS.values():
    split_folder.mkdir(parents=True, exist_ok=True)

split_csvs = {
    "train": "train.csv",
    "val": "val.csv",
    "test": "test.csv",
}

video_ids = []
for csv_path in split_csvs.values():
    video_ids.extend(pd.read_csv(csv_path)["video_id"].tolist())

failed = []
for video_id in tqdm(video_ids, desc="Processing videos"):
    split = _split_lookup[video_id]
    out_path = SPLIT_FOLDERS[split] / f"{video_id}.npy"
    if out_path.exists():
        continue

    try:
        video_path = find_video(video_id)
        frames = load_frames(video_path)
        landmarks = extract_landmarks(frames)
        save_sequence(video_id, landmarks)
    except Exception as e:
        failed.append((video_id, str(e)))

print(f"Done. Saved {len(video_ids) - len(failed)} sequences.")
if failed:
    print(f"Failed: {len(failed)}")
    for video_id, error in failed[:10]:
        print(f"  {video_id}: {error}")

Processing videos:   0%|          | 0/15238 [00:00<?, ?it/s]I0000 00:00:1782828888.337132 32172089 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 11, prefix = pthread-default
I0000 00:00:1782828888.404871 32172089 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 89.3), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1782828888.409169 32172094 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782828888.412878 32172092 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1782828888.426787 32172096 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
Processing videos:   3%|▎         | 395/15238 [02:00<1:19:

Done. Saved 15238 sequences.


E0000 00:00:1782833510.297306 32172090 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-06-30T11:45:50.291066-04:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1782833570.301963 32172090 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-06-30T11:45:50.291066-04:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1782833630.305157 32172090 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-06-30T11:45:50.291066-04:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1782833690.311414 32172090 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not v

Interpolation step (fill in missing gaps) between sequences.

In [ ]:
RAW_DATA_DIR = Path("raw_data")
FINAL_DATA_DIR = Path("final_data")

INTERP_SPLITS = {
    "train": "train",
    "val": "validate",
    "test": "test",
}


def hand_detected_mask(seq):
    """True for frames with at least one non-zero landmark."""
    return np.any(seq != 0, axis=1)


def gesture_window(mask):
    """Return inclusive (start, end) of first/last hand detection, or (None, None)."""
    if not mask.any():
        return None, None
    start = int(np.argmax(mask))
    end = int(len(mask) - 1 - np.argmax(mask[::-1]))
    return start, end


def interpolate_gaps(seq):
    """Linearly fill zero-frame gaps inside the gesture window."""
    seq = seq.copy()
    detected = hand_detected_mask(seq)
    start_win, end_win = gesture_window(detected)
    if start_win is None:
        return seq

    i = start_win
    while i <= end_win:
        if detected[i]:
            i += 1
            continue

        gap_start = i
        while i <= end_win and not detected[i]:
            i += 1
        gap_end = i
        gap = gap_end - gap_start

        if gap <= 0 or gap_end > end_win:
            continue

        prev_frame = seq[gap_start - 1]
        next_frame = seq[gap_end]
        start = gap_start - 1

        for k in range(1, gap + 1):
            alpha = k / (gap + 1)
            seq[start + k] = (1 - alpha) * prev_frame + alpha * next_frame

    return seq

E0000 00:00:1782834230.355976 32172090 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-06-30T11:45:50.291066-04:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1782834290.362700 32172090 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-06-30T11:45:50.291066-04:00
=== Source Location Trace: ===
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


In [5]:
from tqdm import tqdm

for split_name, folder in INTERP_SPLITS.items():
    in_dir = RAW_DATA_DIR / folder
    out_dir = FINAL_DATA_DIR / folder
    out_dir.mkdir(parents=True, exist_ok=True)

    npy_files = sorted(in_dir.glob("*.npy"))
    for npy_path in tqdm(npy_files, desc=f"Interpolating {split_name}"):
        out_path = out_dir / npy_path.name
        if out_path.exists():
            continue

        seq = np.load(npy_path)
        interpolated = interpolate_gaps(seq)
        np.save(out_path, interpolated.astype(np.float32))

print("Interpolation complete.")

Interpolating test: 100%|██████████| 1524/1524 [00:00<00:00, 3833.47it/s]

Interpolation complete.


In [6]:
# Stats: zero frames (no hand detection) per sequence in final_data
label_lookup = {}
for csv_path in ["train.csv", "val.csv", "test.csv"]:
    for _, row in pd.read_csv(csv_path).iterrows():
        label_lookup[int(row["video_id"])] = row["label"]

rows = []
for split, folder in INTERP_SPLITS.items():
    for npy_path in (FINAL_DATA_DIR / folder).glob("*.npy"):
        video_id = int(npy_path.stem)
        seq = np.load(npy_path)
        detected = hand_detected_mask(seq)
        rows.append({
            "video_id": video_id,
            "split": split,
            "label": label_lookup.get(video_id, "UNKNOWN"),
            "zero_frames": int((~detected).sum()),
            "all_zero": not detected.any(),
        })

stats_df = pd.DataFrame(rows)

print(f"Total sequences: {len(stats_df)}")
print(f"\nOverall zero-frame stats (per sequence):")
print(f"  mean:   {stats_df['zero_frames'].mean():.2f}")
print(f"  median: {stats_df['zero_frames'].median():.1f}")
print(
    f"  all-zero sequences: {stats_df['all_zero'].sum()} "
    f"({100 * stats_df['all_zero'].mean():.2f}%)"
)

by_class = (
    stats_df.groupby("label")
    .agg(
        count=("video_id", "count"),
        mean_zero_frames=("zero_frames", "mean"),
        median_zero_frames=("zero_frames", "median"),
        total_zero_frames=("zero_frames", "sum"),
        all_zero_count=("all_zero", "sum"),
    )
    .assign(all_zero_pct=lambda d: 100 * d["all_zero_count"] / d["count"])
    .sort_values("count", ascending=False)
)

print("\nPer-class breakdown:")
display(
    by_class.round({"mean_zero_frames": 2, "median_zero_frames": 1, "all_zero_pct": 2})
)

Total sequences: 15238

Overall zero-frame stats (per sequence):
  mean:   21.43
  median: 22.0
  all-zero sequences: 3089 (20.27%)

Per-class breakdown:


,count,mean_zero_frames,median_zero_frames,total_zero_frames,all_zero_count,all_zero_pct
label,,,,,,
No gesture,7187,22.00,23.0,158125,2610,36.32
Swiping Down,2072,21.33,23.0,44204,73,3.52
Swiping Left,2009,21.04,21.0,42278,166,8.26
Swiping Up,2009,19.54,20.0,39261,89,4.43
Swiping Right,1961,21.78,23.0,42707,151,7.70


Remove all-zero sequences from `final_data` for swipe gestures only (keep all-zero `No gesture` samples).

In [7]:
SWIPE_LABELS = {
    "Swiping Down",
    "Swiping Left",
    "Swiping Up",
    "Swiping Right",
}

label_lookup = {}
for csv_path in ["train.csv", "val.csv", "test.csv"]:
    for _, row in pd.read_csv(csv_path).iterrows():
        label_lookup[int(row["video_id"])] = row["label"]

deleted = []
for split, folder in INTERP_SPLITS.items():
    for npy_path in (FINAL_DATA_DIR / folder).glob("*.npy"):
        video_id = int(npy_path.stem)
        label = label_lookup.get(video_id)
        if label not in SWIPE_LABELS:
            continue

        seq = np.load(npy_path)
        if hand_detected_mask(seq).any():
            continue

        npy_path.unlink()
        deleted.append({"video_id": video_id, "split": split, "label": label})

deleted_df = pd.DataFrame(deleted)
print(f"Deleted {len(deleted_df)} all-zero swipe sequences from final_data.")
if not deleted_df.empty:
    print("\nBy label:")
    print(deleted_df.groupby("label").size().to_string())
    print("\nBy split:")
    print(deleted_df.groupby("split").size().to_string())

Deleted 479 all-zero swipe sequences from final_data.

By label:
label
Swiping Down      73
Swiping Left     166
Swiping Right    151
Swiping Up        89

By split:
split
test      54
train    393
val       32
